# Bitcoin Price Prediction — Advanced

**Tabular Regression Project** — Predict Bitcoin price with advanced feature engineering.

Models: CatBoost · LightGBM · XGBoost · FLAML · LazyPredict
Dataset: Bitcoin Historical (367 rows × 2 columns, enriched via feature engineering)
Target: `Price`

## 2 · Project Overview

This is an **advanced version** of the Bitcoin Price Prediction project. We use the same small dataset but apply more sophisticated feature engineering: technical indicators (RSI, Bollinger Bands, MACD-like features), more lag windows, and percentage returns. The goal is to push tabular ML as far as possible on minimal data.

## 3 · Learning Objectives

1. Compute technical indicators (RSI, Bollinger Bands) from price data.
2. Engineer advanced lag and rolling features.
3. Understand diminishing returns of feature engineering on small data.
4. Compare model performance with basic vs advanced features.
5. Use LazyPredict and FLAML for benchmarking.

## 4 · Problem Statement

Predict Bitcoin's daily **Price** using only historical price data, enhanced with technical analysis features. Demonstrate how advanced feature engineering can (or cannot) improve predictions on tiny datasets.

## 5 · Why This Project Matters

- Shows how far feature engineering alone can take you.
- Introduces financial technical indicators used by traders.
- Highlights the importance of **data quantity** vs. **feature quality**.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 367 (raw), ~320 after feature engineering |
| **Original Columns** | 2 (Date, Price) |
| **Engineered Features** | 20+ |
| **Target** | Price (continuous, USD) |
| **File** | `bitcoin.csv` (from sibling folder) |

## 7 · Dataset Source and License Notes

- **Source**: Same dataset as the basic Bitcoin Price Prediction project.
- **License**: Public domain / educational use.
- **Note**: This advanced project reuses the same CSV but applies richer feature engineering.

## 8 · Environment Setup

In [ ]:
import subprocess, sys

def _install_if_missing(pkg, import_name=None):
    try:
        __import__(import_name or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg, imp in [('catboost', None), ('lightgbm', None), ('xgboost', None),
                 ('flaml', None), ('lazypredict', None)]:
    _install_if_missing(pkg, imp)

print('All packages ready.')

## 9 · Imports

In [ ]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred): return mean_squared_error(y_true, y_pred, squared=False)

warnings.filterwarnings('ignore')
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('Imports complete.')

## 10 · Configuration / Constants

In [ ]:
SEED = 42
TEST_SIZE = 0.2
TARGET = 'Price'
np.random.seed(SEED)

## 11 · Dataset Download or Loading

In [ ]:
# Try local first, then sibling folder
DATA_PATH = os.path.join(SAVE_DIR, 'bitcoin.csv')
if not os.path.exists(DATA_PATH):
    DATA_PATH = os.path.join(SAVE_DIR, '..', 'Bitcoin Price Prediction', 'bitcoin.csv')
assert os.path.exists(DATA_PATH), f'Dataset not found at {DATA_PATH}'
df = pd.read_csv(DATA_PATH)
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)
print(f'Loaded: {df.shape}')
df.head()

## 12 · Data Validation Checks

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print(f'\nDuplicates: {df.duplicated().sum()}')
print(f'Date range: {df["Date"].min()} to {df["Date"].max()}')
print(f'Price range: [{df[TARGET].min():.2f}, {df[TARGET].max():.2f}]')

## 13 · Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].plot(df['Date'], df[TARGET])
axes[0,0].set_title('Bitcoin Price Over Time')

df[TARGET].hist(bins=30, ax=axes[0,1], edgecolor='black')
axes[0,1].set_title('Price Distribution')

df[TARGET].pct_change().hist(bins=50, ax=axes[1,0], edgecolor='black')
axes[1,0].set_title('Daily Returns Distribution')

df[TARGET].rolling(30).std().plot(ax=axes[1,1])
axes[1,1].set_title('30-Day Rolling Volatility')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## 14 · Target Analysis

In [ ]:
print(df[TARGET].describe())
print(f'\nSkewness: {df[TARGET].skew():.2f}')
print(f'Kurtosis: {df[TARGET].kurtosis():.2f}')

## 15 · Train / Validation / Test Split Strategy

Random 80/20 split for demonstration. A time-based split would be more appropriate for production.

## 16 · Preprocessing Strategy

No scaling needed for tree models. All features are engineered from the Price column.

## 17 · Feature Engineering (Advanced)

We engineer:
- Temporal features from Date
- Lag features (1, 3, 7, 14, 30 days)
- Rolling statistics (mean, std, min, max)
- Returns and log returns
- Technical indicators: RSI, Bollinger Bands, EMA ratios

In [ ]:
# Temporal features
df['year'] = df['Date'].dt.year
df['month'] = df['Date'].dt.month
df['day'] = df['Date'].dt.day
df['dayofweek'] = df['Date'].dt.dayofweek
df['dayofyear'] = df['Date'].dt.dayofyear

# Lag features
for lag in [1, 3, 7, 14, 30]:
    df[f'price_lag_{lag}'] = df[TARGET].shift(lag)

# Rolling statistics (shifted to avoid leakage)
for window in [7, 14, 30]:
    shifted = df[TARGET].shift(1)
    df[f'roll_mean_{window}'] = shifted.rolling(window).mean()
    df[f'roll_std_{window}'] = shifted.rolling(window).std()
    df[f'roll_min_{window}'] = shifted.rolling(window).min()
    df[f'roll_max_{window}'] = shifted.rolling(window).max()

# Returns
df['return_1d'] = df[TARGET].pct_change(1)
df['return_7d'] = df[TARGET].pct_change(7)
df['log_return'] = np.log(df[TARGET] / df[TARGET].shift(1))

# RSI (14-day)
delta = df[TARGET].diff()
gain = delta.where(delta > 0, 0).rolling(14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
rs = gain / loss.replace(0, np.nan)
df['rsi_14'] = 100 - (100 / (1 + rs))

# Bollinger Band position
bb_mean = df[TARGET].shift(1).rolling(20).mean()
bb_std = df[TARGET].shift(1).rolling(20).std()
df['bb_position'] = (df[TARGET].shift(1) - bb_mean) / bb_std.replace(0, np.nan)

# EMA ratios
df['ema_12'] = df[TARGET].shift(1).ewm(span=12).mean()
df['ema_26'] = df[TARGET].shift(1).ewm(span=26).mean()
df['ema_ratio'] = df['ema_12'] / df['ema_26'].replace(0, np.nan)

df = df.dropna().reset_index(drop=True)
print(f'Shape after feature engineering: {df.shape}')

feature_cols = [c for c in df.columns if c not in [TARGET, 'Date']]
X = df[feature_cols]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Features ({len(feature_cols)}): {feature_cols}')

## 18 · Baseline Model

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)

baseline_rmse = root_mean_squared_error(y_test, y_pred_base)
baseline_mae = mean_absolute_error(y_test, y_pred_base)
baseline_r2 = r2_score(y_test, y_pred_base)

print(f'Baseline Linear Regression:')
print(f'  RMSE: {baseline_rmse:.2f}')
print(f'  MAE:  {baseline_mae:.2f}')
print(f'  R²:   {baseline_r2:.4f}')

## 19 · LazyPredict Benchmark

In [ ]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print(lazy_models.head(15))

## 20 · FLAML AutoML Run

In [ ]:
try:
    from flaml import AutoML
    flaml_model = AutoML()
    flaml_model.fit(
        X_train, y_train,
        task='regression',
        time_budget=60,
        metric='rmse',
        seed=SEED,
        verbose=0
    )
    y_pred_flaml = flaml_model.predict(X_test)
    flaml_rmse = root_mean_squared_error(y_test, y_pred_flaml)
    flaml_mae = mean_absolute_error(y_test, y_pred_flaml)
    flaml_r2 = r2_score(y_test, y_pred_flaml)
    print(f'FLAML best: {flaml_model.best_estimator}')
    print(f'  RMSE: {flaml_rmse:.2f}')
    print(f'  MAE:  {flaml_mae:.2f}')
    print(f'  R²:   {flaml_r2:.4f}')
except Exception as e:
    print(f'FLAML failed: {e}')
    flaml_rmse = flaml_mae = flaml_r2 = None

## 21 · Additional Best-Library Workflow (CatBoost + LightGBM + XGBoost)

In [ ]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

models = {
    'CatBoost': CatBoostRegressor(iterations=300, learning_rate=0.1, depth=4,
                                   random_seed=SEED, verbose=0),
    'LightGBM': LGBMRegressor(n_estimators=300, learning_rate=0.1, max_depth=4,
                              random_state=SEED, verbose=-1),
    'XGBoost': XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=4,
                            random_state=SEED, verbosity=0)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'RMSE': root_mean_squared_error(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds)
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.2f}, MAE={results[name]["MAE"]:.2f}, R²={results[name]["R2"]:.4f}')

## 22 · Top 3 Model Selection

In [ ]:
all_results = {}
all_results['Baseline_LR'] = {'RMSE': baseline_rmse, 'MAE': baseline_mae, 'R2': baseline_r2}
if flaml_r2 is not None:
    all_results['FLAML'] = {'RMSE': flaml_rmse, 'MAE': flaml_mae, 'R2': flaml_r2}
all_results.update(results)

ranked = sorted(all_results.items(), key=lambda x: x[1]['RMSE'])
print('All models ranked by RMSE:')
for name, m in ranked:
    print(f"  {name:20s}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  R²={m['R2']:.4f}")

top3_names = [r[0] for r in ranked[:3]]
print(f'\nTop 3: {top3_names}')

## 23 · Final Training and Evaluation of Top 3

In [ ]:
print('Top 3 Final Results:')
print('=' * 60)
for name in top3_names:
    m = all_results[name]
    print(f"{name}:")
    print(f"  RMSE: {m['RMSE']:.2f}")
    print(f"  MAE:  {m['MAE']:.2f}")
    print(f"  R²:   {m['R2']:.4f}")
    print()

## 24 · Error Analysis

In [ ]:
best_name = top3_names[0]
if best_name in models:
    best_model = models[best_name]
else:
    best_model = models['CatBoost']

y_pred_best = best_model.predict(X_test)
residuals = y_test.values - y_pred_best

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_pred_best, residuals, alpha=0.5)
axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Predicted')

axes[1].hist(residuals, bins=20, edgecolor='black')
axes[1].set_title('Residual Distribution')

axes[2].scatter(y_test, y_pred_best, alpha=0.5)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted')
axes[2].set_title('Actual vs Predicted')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100)
plt.show()

## 25 · Interpretation and Business Insight

- **Lag features dominate**: Recent prices are the strongest predictors.
- **Technical indicators** (RSI, Bollinger, EMA) add marginal value on such a small dataset.
- Advanced feature engineering helps but cannot overcome the fundamental lack of data.
- The autocorrelation in price data makes lag features almost 'cheating' on random splits.

## 26 · Limitations

- Same dataset as the basic project — only 367 rows.
- Many engineered features from few original columns risks overfitting.
- Random split overstates model quality.
- No external data sources.

## 27 · How to Improve This Project

1. Use proper walk-forward validation.
2. Add order-book and volume data.
3. Incorporate on-chain metrics.
4. Test with larger date ranges.
5. Combine with sentiment analysis (Twitter/Reddit).

## 28 · Production Considerations

- Feature engineering pipeline must be reproducible in real-time.
- Latency: rolling/lag features need efficient computation.
- Monitoring: crypto regime changes can invalidate all learned patterns.
- Risk management: never deploy without position sizing and stop-losses.

## 29 · Common Mistakes

1. **Feature leakage**: Not shifting rolling features by 1 day.
2. **Too many features for few rows**: Overfitting risk.
3. **Ignoring regime changes**: Crypto markets shift between bull/bear.
4. **Random splits for time data**: Inflates metrics.

## 30 · Mini Challenge / Exercises

1. Compare the basic vs. advanced feature set — which gives better R²?
2. Implement walk-forward validation with 30-day test windows.
3. Add a 'naive baseline' that predicts yesterday's price.
4. Calculate feature importance and remove the bottom 50% of features.

## 31 · Final Summary / Key Takeaways

- Advanced feature engineering provides marginal improvement on this small dataset.
- Technical indicators (RSI, Bollinger, EMA) are useful concepts but need more data to shine.
- The main lesson: **data quantity matters more than feature complexity**.
- Always validate financial models with time-proper splits.

In [ ]:
metrics = {}
for name in top3_names:
    metrics[name] = all_results[name]
metrics['baseline_linear_regression'] = all_results['Baseline_LR']

with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to metrics.json')
print('\nNotebook complete.')